# Voice Cloning Detector — Training Notebook (SIH26104)

Yeh notebook Google Colab mein chalana hai.

**Kya karna hai (order mein):**
1. Runtime -> Change runtime type -> **T4 GPU** select karo (training tez hogi)
2. Cell 1 se libraries install karo
3. Apni `real/` aur `fake/` clips ka ek ZIP banao is structure mein:
```
clips.zip
├── real/
│   ├── real1.wav
│   ├── real2.wav
│   └── ...
└── fake/
    ├── fake1.wav
    ├── fake2.wav
    └── ...
```
4. Cell 2 chalao, aur jab poocha jaaye tab `clips.zip` upload karo
5. Baaki cells ek ek karke chalate jao (Shift+Enter)
6. Last cell mein `model.pkl` aur `scaler.pkl` do files download honge — yeh hi files Antigravity wale app mein daalni hain


In [ ]:
# Cell 1 — install libraries
!pip install -q librosa soundfile scikit-learn joblib
print("done")


In [ ]:
# Cell 2 — upload your clips.zip (real/ and fake/ folders inside)
from google.colab import files
import zipfile, os

uploaded = files.upload()          # choose clips.zip when prompted
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('data')

for root, dirs, fs in os.walk('data'):
    if fs:
        print(root, '->', len(fs), 'files')


In [ ]:
# Cell 3 — feature extraction (SAME logic as features.py used by the Flask app)
# If you change anything here, copy the same change into features.py before running the app.

import numpy as np
import librosa

SAMPLE_RATE = 16000

def load_audio(path):
    y, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y, _ = librosa.effects.trim(y, top_db=25)
    if len(y) < SAMPLE_RATE:
        y = np.pad(y, (0, SAMPLE_RATE - len(y)))
    return y, sr

def extract_features(y, sr=SAMPLE_RATE):
    feats = []
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    feats += list(np.mean(mfcc, axis=1)); feats += list(np.std(mfcc, axis=1))
    delta = librosa.feature.delta(mfcc)
    feats += list(np.mean(delta, axis=1)); feats += list(np.std(delta, axis=1))
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    feats += [np.mean(centroid), np.std(centroid)]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    feats += [np.mean(bandwidth), np.std(bandwidth)]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    feats += [np.mean(rolloff), np.std(rolloff)]
    flatness = librosa.feature.spectral_flatness(y=y)
    feats += [np.mean(flatness), np.std(flatness)]
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    feats += list(np.mean(contrast, axis=1))
    zcr = librosa.feature.zero_crossing_rate(y)
    feats += [np.mean(zcr), np.std(zcr)]
    rms = librosa.feature.rms(y=y)
    feats += [np.mean(rms), np.std(rms)]
    f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0_voiced = f0[voiced_flag] if voiced_flag is not None else np.array([])
    if len(f0_voiced) > 1:
        feats += [np.nanmean(f0_voiced), np.nanstd(f0_voiced)]
    else:
        feats += [0.0, 0.0]
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    feats += list(np.mean(chroma, axis=1))
    return np.array(feats, dtype=np.float32)

print("feature function ready")


In [ ]:
# Cell 4 — build the dataset (X = features, y = labels) from data/real and data/fake
import os

X, y_labels, file_list = [], [], []

for label, folder in [(0, 'data/real'), (1, 'data/fake')]:   # 0 = REAL, 1 = FAKE
    if not os.path.isdir(folder):
        print('missing folder:', folder); continue
    for fname in os.listdir(folder):
        if not fname.lower().endswith(('.wav', '.mp3', '.m4a', '.ogg')):
            continue
        path = os.path.join(folder, fname)
        try:
            audio, sr = load_audio(path)
            feat = extract_features(audio, sr)
            X.append(feat)
            y_labels.append(label)
            file_list.append(path)
        except Exception as e:
            print('skipped', path, '-', e)

import numpy as np
X = np.array(X)
y_labels = np.array(y_labels)
print('Total samples:', len(X), '| Real:', sum(y_labels==0), '| Fake:', sum(y_labels==1))
print('Feature vector length:', X.shape[1] if len(X) else 'no data')


In [ ]:
# Cell 5 — split into train/test, scale features, train classifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_labels, test_size=0.25, random_state=42, stratify=y_labels
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = RandomForestClassifier(
    n_estimators=300, max_depth=12, class_weight='balanced', random_state=42
)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
print('Test accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['REAL', 'FAKE']))
print('Confusion matrix (rows=actual, cols=predicted):')
print(confusion_matrix(y_test, y_pred))


In [ ]:
# Cell 6 — IMPORTANT reality check before you trust the demo
# This tells you if the model might just be learning "phone recording vs TTS file"
# instead of real deepfake artifacts. Read the printed advice.

n_real, n_fake = sum(y_labels == 0), sum(y_labels == 1)
print(f"Real samples: {n_real} | Fake samples: {n_fake}")

if len(X) < 40:
    print("\n⚠️  WARNING: Very few samples. Accuracy above is not reliable.")
    print("Cut your clips into 3-4 second chunks (see note below) to multiply your data.")

if n_real < 5 or n_fake < 5:
    print("\n⚠️  WARNING: You need at least 10-15 real and 10-15 fake clips (or chunks) for a")
    print("believable demo. Fewer than this, and one lucky test split can look like 95% accuracy")
    print("while the model has actually memorised nothing useful.")

print('''
TIP — turning few long clips into many samples:
Cut every original clip into 3-second pieces with ffmpeg before zipping, e.g.:
  ffmpeg -i original.wav -f segment -segment_time 3 -c copy chunk_%02d.wav
Put all chunk_*.wav files into real/ or fake/ instead of the one long file.
''')


In [ ]:
# Cell 7 — save model + scaler, and download them
import joblib

joblib.dump(clf, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')

from google.colab import files
files.download('model.pkl')
files.download('scaler.pkl')

print("Downloaded model.pkl and scaler.pkl — put both files in the app's model/ folder.")
